This notebook is meant to be run on the JHU Rockfish cluster, using COMPAS data files that were produced and saved on the cluster (as in `pop_paths`). Once run, the output is saved to a more manageable set of csv files in `data_path` that can be used for all the plots in the paper.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.options.mode.chained_assignment = None 

import h5py as h5 
from astropy import units as u
from astropy import constants as c
from astropy.cosmology import Planck18
from scipy.interpolate import interp1d

import os
import scipy

from collections import Counter
from collections import defaultdict
import gc

# from sklearn.utils import resample

pd.options.display.max_columns = None

In [2]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib
from matplotlib.ticker import LogLocator, AutoMinorLocator, MultipleLocator
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec

fontparams = {
    "font.family": "serif",
    "mathtext.fontset" : "stix",
    "grid.color": "gray",
    "grid.linestyle": ":",
    "axes.titlesize": "18",
    "axes.labelsize": "16",
    "xtick.labelsize": "16",
    "ytick.labelsize": "16",
    "xtick.labelbottom": "True",
    "legend.framealpha": "1",
}
rcParams.update(fontparams) 

%config InlineBackend.figure_format='retina' # very useful command for high-res images

from cycler import cycler

colorPalette = {'red': "#E64D4E",
                'orange': "#EE9063",
                'yellow': "#FFDD7B",
                'green': "#77AC54",
                'blue': "#0B92B1",
                'violet': "#665191",
                'gray': "#B4B4B4"
}

custom_cycler = (cycler(color=[colorPalette['red'], colorPalette['blue'], colorPalette['green']]))


In [ ]:
pop_names = ['notides', 'realistic', 'perfect', 'z77']

pop_paths = {
            'notides': 
                ['../pop_sims/notides_pop_v329_combined.h5'],
            
            'realistic':
                ['../pop_sims/realistic_pop_v329_combined.h5'],
            
            'perfect':
                ['../pop_sims/perfect_pop_v329_combined.h5'],
             
            'z77':
                ['../pop_sims/z77_pop_v329_combined.h5'] 
            }

pop_labels = {'notides': 'LEGACY',
              'realistic': 'KAPIL26',
              'perfect': 'PERFECT',
              'z77': 'Z77'}

pop_colors = {'notides': colorPalette['red'],
              'realistic': colorPalette['blue'],
              'perfect': colorPalette['green'],
             'z77': colorPalette['orange']}

pop_cmaps = {'notides': 'Reds',
              'realistic': 'Blues',
              'perfect': 'Greens'}

dco_types = ['BBH', 'BHNS', 'BNS']
dco_st = [28, 27, 26]
mt_labels = [
    "No MT", "Stable MT 1→2", "Stable MT 2→1", 
    "CE_Primary", "CE_Secondary", "CE_Both", "MT to Merger"
]
branching_labels = {1: 'Primary', 2: 'Secondary', 3: 'Both', 4: 'MT_to_Merger'}

plot_path = 'pop_plots/'

data_path = 'data_files/'

if not os.path.exists(data_path):
   os.makedirs(data_path)

# Load and Process COMPAS data

In [ ]:
def read_zams(fdata):
    g = fdata['BSE_System_Parameters']
    return {
        'SEED':               g['SEED'][...].squeeze(),
        'Mass@ZAMS(1)':       g['Mass@ZAMS(1)'][...].squeeze(),
        'Mass@ZAMS(2)':       g['Mass@ZAMS(2)'][...].squeeze(),
        'Metallicity@ZAMS(1)':g['Metallicity@ZAMS(1)'][...].squeeze(),
        'Omega(1)':           g['Omega(1)'][...].squeeze(),
        'Omega(2)':           g['Omega(2)'][...].squeeze(),
        'Eccentricity@ZAMS':  g['Eccentricity@ZAMS'][...].squeeze(),
        'SemiMajorAxis@ZAMS': g['SemiMajorAxis@ZAMS'][...].squeeze(),
        'CH_on_MS(1)':        g['CH_on_MS(1)'][...].squeeze().astype(bool),
        'CH_on_MS(2)':        g['CH_on_MS(2)'][...].squeeze().astype(bool),
    }

def read_dco(fdata):
    g = fdata['BSE_Double_Compact_Objects']
    m1 = g['Mass(1)'][...].squeeze()
    m2 = g['Mass(2)'][...].squeeze()
    return {
        'SEED':              g['SEED'][...].squeeze(),
        'Mass@DCO(1)':       m1,
        'Mass@DCO(2)':       m2,
        'SemiMajorAxis@DCO': g['SemiMajorAxis@DCO'][...].squeeze(),
        'Eccentricity@DCO':  g['Eccentricity@DCO'][...].squeeze(),
        'Stellar_Type(1)':   g['Stellar_Type(1)'][...].squeeze(),
        'Stellar_Type(2)':   g['Stellar_Type(2)'][...].squeeze(),
        'Merges_Hubble_Time':g['Merges_Hubble_Time'][...].squeeze().astype(bool),
        'Time@DCO':          g['Time'][...].squeeze(),
        'Coalescence_Time':   g['Coalescence_Time'][...].squeeze(),
        'M_min@DCO':         np.minimum(m1, m2),
        'M_max@DCO':         np.maximum(m1, m2),
    }

def read_sn(fdata):
    g = fdata['BSE_Supernovae']
    sma   = g['SemiMajorAxis<SN'][...].squeeze()
    m_tot = g['Mass_Total@CO(SN)'][...].squeeze()
    m_cp  = g['Mass(CP)'][...].squeeze()

    mt_raw = g['MT_Donor_Hist(SN)'][...].squeeze().astype(str)
    mt_stripped = np.char.strip(mt_raw)
    mt_parsed = [
        list(map(int, s.split('-'))) if s != 'NA' else np.nan
        for s in mt_stripped
    ]
    n_rlof = [len(x) if isinstance(x, list) else 0 for x in mt_parsed]

    omega_orb = np.sqrt(
        c.G * (m_tot + m_cp) * u.M_sun / (sma * u.AU)**3
    ).to(1/u.d)

    return {
        'SEED':                    g['SEED'][...].squeeze(),
        'Time@SN':                 g['Time'][...].squeeze(),
        'RLOF':                    g['Experienced_RLOF(SN)'][...].squeeze().astype(bool),
        'Supernova_State':         g['Supernova_State'][...].squeeze(),
        'SN_Type(SN)':             g['SN_Type(SN)'][...].squeeze(),
        'Stellar_Type_Prev(SN)':   g['Stellar_Type_Prev(SN)'][...].squeeze(),
        'Mass_CO_Core@CO(SN)':     g['Mass_CO_Core@CO(SN)'][...].squeeze(),
        'Mass(SN)':                g['Mass(SN)'][...].squeeze(),
        'Mass(CP)':                m_cp,
        'Omega(SN)':               g['Omega(SN)'][...].squeeze(),
        'SemiMajorAxis<SN':        sma,
        'Mass_Total@CO(SN)':       m_tot,
        'Mass_Core@CO(SN)':        g['Mass_Core@CO(SN)'][...].squeeze(),
        'Radius_Total@CO(SN)':     g['Radius_Total@CO(SN)'][...].squeeze(),
        'Radius_Core@CO(SN)':      g['Radius_Core@CO(SN)'][...].squeeze(),
        'MT_Donor_Hist(SN)':       mt_parsed,
        'Ang_Momentum(SN)':        g['Ang_Momentum(SN)'][...].squeeze(),
        'SN_Orbit_Inclination_Angle': g['SN_Orbit_Inclination_Angle'][...].squeeze(),
        'Number_of_RLOF(SN)':      n_rlof,
        'Orbital_Period<SN':       (2 * np.pi / omega_orb).value,
    }

def read_ce(fdata):
    g = fdata['BSE_Common_Envelopes']
    return {
        'SEED':               g['SEED'][...].squeeze(),
        'MT_History':         g['MT_History'][...].squeeze(),
        'SemiMajorAxis<CE':   g['SemiMajorAxis<CE'][...].squeeze(),
        'Eccentricity<CE':    g['Eccentricity<CE'][...].squeeze(),
        'Mass(1)<CE':         g['Mass(1)<CE'][...].squeeze(),
        'Mass(2)<CE':         g['Mass(2)<CE'][...].squeeze(),
        'Stellar_Type(1)<CE': g['Stellar_Type(1)<CE'][...].squeeze(),
        'Stellar_Type(2)<CE': g['Stellar_Type(2)<CE'][...].squeeze(),
        'Stellar_Type(1)>CE': g['Stellar_Type(1)'][...].squeeze(),
        'Stellar_Type(2)>CE': g['Stellar_Type(2)'][...].squeeze(),
        'Time@CE':            g['Time'][...].squeeze(),
    }


def process_file(pop_path):
    """Read one HDF5 file and return a merged, DCO-filtered DataFrame."""
    with h5.File(pop_path, 'r') as fdata:
        zams = pd.DataFrame(read_zams(fdata))
        dco  = pd.DataFrame(read_dco(fdata))
        sn   = pd.DataFrame(read_sn(fdata))
        ce   = pd.DataFrame(read_ce(fdata))

    # Filter to DCO systems immediately to shed most rows
    valid_seeds = dco['SEED'].values
    zams = zams[zams['SEED'].isin(valid_seeds)]
    sn   = sn[sn['SEED'].isin(valid_seeds)]
    ce   = ce[ce['SEED'].isin(valid_seeds)]

    merged = (
        zams
        .merge(dco, on='SEED', how='inner')   # inner = only DCO systems
        .merge(sn,  on='SEED', how='left')
        .merge(ce,  on='SEED', how='left')
    )
    del zams, dco, sn, ce
    gc.collect()
    return merged


def pivot_and_classify(pop_df, branching_labels):
    ce_columns = [
        'SEED','MT_History','SemiMajorAxis<CE','Eccentricity<CE',
        'Mass(1)<CE','Mass(2)<CE','Stellar_Type(1)<CE','Stellar_Type(2)<CE',
        'Stellar_Type(1)>CE','Stellar_Type(2)>CE','Time@CE',
    ]
    sn_columns = [
        'SEED','Time@SN','RLOF','Supernova_State','SN_Type(SN)',
        'Stellar_Type_Prev(SN)','Mass_CO_Core@CO(SN)','Mass(SN)','Mass(CP)',
        'Omega(SN)','SemiMajorAxis<SN','Mass_Total@CO(SN)','Mass_Core@CO(SN)',
        'Radius_Total@CO(SN)','Radius_Core@CO(SN)','MT_Donor_Hist(SN)',
        'Ang_Momentum(SN)','SN_Orbit_Inclination_Angle','Number_of_RLOF(SN)',
        'Orbital_Period<SN',
    ]

    # --- CE pivot ---
    ce_mask = pop_df['MT_History'].between(3, 5)
    ce_pivot = (
        pop_df.loc[ce_mask, ce_columns]
        .pivot_table(index='SEED', columns='MT_History')
    )
    ce_pivot.columns = [
        f'{col[0]}_{branching_labels[int(col[1])-2]}' for col in ce_pivot.columns
    ]
    mt_counts = (
        pop_df.loc[ce_mask, ['SEED', 'MT_History', 'Time@CE']]
        .pivot_table(index='SEED', columns='MT_History',
                     values='Time@CE', aggfunc='count', fill_value=0)
        .astype(bool)
    )
    mt_counts.columns = [f'CE_{branching_labels[int(c)-2]}' for c in mt_counts.columns]
    ce_pivot = ce_pivot.join(mt_counts)
    del mt_counts

    # --- SN pivot ---
    sn_pivot = (
        pop_df[sn_columns]
        .pivot_table(index='SEED', columns='Supernova_State', aggfunc='first')
    )
    sn_pivot.columns = [
        f'{col[0]}_{branching_labels[int(col[1])]}' for col in sn_pivot.columns
    ]
    for col in ('RLOF_Primary', 'RLOF_Secondary', 'RLOF_Both'):
        if col in sn_pivot.columns:
            sn_pivot[col] = sn_pivot[col].astype(bool)

    # --- Combine ---
    original_cols = [
        c for c in pop_df.columns
        if c not in set(sn_columns + ce_columns) or c == 'SEED'
    ]
    result = (
        pop_df[original_cols]
        .drop_duplicates(subset=['SEED'])
        .join(sn_pivot, on='SEED', how='left')
        .join(ce_pivot, on='SEED', how='left')
    )
    del sn_pivot, ce_pivot
    gc.collect()

    st1, st2 = result['Stellar_Type(1)'], result['Stellar_Type(2)']
    result['BBH']  = (st1 + st2 == 28)
    result['BHNS'] = (st1 + st2 == 27)
    result['BNS']  = (st1 + st2 == 26)
    return result


In [ ]:
# ── Main loop ────────────────────────────────────────────────────────────────
pop_dfs = {}

for pop_name in pop_names:
    print(pop_name)
    chunks = []

    for pop_path in pop_paths[pop_name]:
        chunks.append(process_file(pop_path))
        gc.collect()

    pop_df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    pop_dfs[pop_name] = pivot_and_classify(pop_df, branching_labels)
    del pop_df
    gc.collect()

    print()

notides


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '../pop_sims/notides_pop_v329_combined.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# Temporary stop gap, in case CE data is not present in some populations
for pop_name in pop_names:
    if 'CE_Both' not in pop_dfs[pop_name]:
        pop_dfs[pop_name]['CE_Both'] = False
    if 'CE_Primary' not in pop_dfs[pop_name]:
        pop_dfs[pop_name]['CE_Primary'] = False
    if 'CE_Secondary' not in pop_dfs[pop_name]:
        pop_dfs[pop_name]['CE_Secondary'] = False
    pop_dfs[pop_name]['CE_Primary'] = pop_dfs[pop_name]['CE_Primary'].fillna(False)   
    pop_dfs[pop_name]['CE_Secondary'] = pop_dfs[pop_name]['CE_Secondary'].fillna(False)
    pop_dfs[pop_name]['CE_Both'] = pop_dfs[pop_name]['CE_Both'].fillna(False)

/tmp/ipykernel_2382825/1353156457.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pop_dfs[pop_name]['CE_Primary'] = pop_dfs[pop_name]['CE_Primary'].fillna(False)
/tmp/ipykernel_2382825/1353156457.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pop_dfs[pop_name]['CE_Secondary'] = pop_dfs[pop_name]['CE_Secondary'].fillna(False)
/tmp/ipykernel_2382825/1353156457.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in t

,SEED,Mass@ZAMS(1),Mass@ZAMS(2),Metallicity@ZAMS(1),Omega(1),Omega(2),Eccentricity@ZAMS,SemiMajorAxis@ZAMS,CH_on_MS(1),CH_on_MS(2),Mass@DCO(1),Mass@DCO(2),SemiMajorAxis@DCO,Eccentricity@DCO,Stellar_Type(1),Stellar_Type(2),Merges_Hubble_Time,Time@DCO,Coalescence_Time,M_min@DCO,M_max@DCO,Ang_Momentum(SN)_Primary,Ang_Momentum(SN)_Secondary,Ang_Momentum(SN)_Both,MT_Donor_Hist(SN)_Primary,MT_Donor_Hist(SN)_Secondary,Mass(CP)_Primary,Mass(CP)_Secondary,Mass(CP)_Both,Mass(SN)_Primary,Mass(SN)_Secondary,Mass(SN)_Both,Mass_CO_Core@CO(SN)_Primary,Mass_CO_Core@CO(SN)_Secondary,Mass_CO_Core@CO(SN)_Both,Mass_Core@CO(SN)_Primary,Mass_Core@CO(SN)_Secondary,Mass_Core@CO(SN)_Both,Mass_Total@CO(SN)_Primary,Mass_Total@CO(SN)_Secondary,Mass_Total@CO(SN)_Both,Number_of_RLOF(SN)_Primary,Number_of_RLOF(SN)_Secondary,Number_of_RLOF(SN)_Both,Omega(SN)_Primary,Omega(SN)_Secondary,Omega(SN)_Both,Orbital_Period<SN_Primary,Orbital_Period<SN_Secondary,Orbital_Period<SN_Both,RLOF_Primary,RLOF_Secondary,RLOF_Both,Radius_Core@CO(SN)_Primary,Radius_Core@CO(SN)_Secondary,Radius_Core@CO(SN)_Both,Radius_Total@CO(SN)_Primary,Radius_Total@CO(SN)_Secondary,Radius_Total@CO(SN)_Both,SN_Orbit_Inclination_Angle_Primary,SN_Orbit_Inclination_Angle_Secondary,SN_Orbit_Inclination_Angle_Both,SN_Type(SN)_Primary,SN_Type(SN)_Secondary,SN_Type(SN)_Both,SemiMajorAxis<SN_Primary,SemiMajorAxis<SN_Secondary,SemiMajorAxis<SN_Both,Stellar_Type_Prev(SN)_Primary,Stellar_Type_Prev(SN)_Secondary,Stellar_Type_Prev(SN)_Both,Time@SN_Primary,Time@SN_Secondary,Time@SN_Both,Eccentricity<CE_Primary,Eccentricity<CE_Secondary,Eccentricity<CE_Both,Mass(1)<CE_Primary,Mass(1)<CE_Secondary,Mass(1)<CE_Both,Mass(2)<CE_Primary,Mass(2)<CE_Secondary,Mass(2)<CE_Both,SemiMajorAxis<CE_Primary,SemiMajorAxis<CE_Secondary,SemiMajorAxis<CE_Both,Stellar_Type(1)<CE_Primary,Stellar_Type(1)<CE_Secondary,Stellar_Type(1)<CE_Both,Stellar_Type(1)>CE_Primary,Stellar_Type(1)>CE_Secondary,Stellar_Type(1)>CE_Both,Stellar_Type(2)<CE_Primary,Stellar_Type(2)<CE_Secondary,Stellar_Type(2)<CE_Both,Stellar_Type(2)>CE_Primary,Stellar_Type(2)>CE_Secondary,Stellar_Type(2)>CE_Both,Time@CE_Primary,Time@CE_Secondary,Time@CE_Both,CE_Primary,CE_Secondary,CE_Both,BBH,BHNS,BNS
0,52002,35.525470,25.347890,0.004520,1.066646e-03,2.011513e+04,0.604388,5.371855,False,False,10.842544,9.366588,0.020569,2.220446e-16,14,14,True,8.940780,2.804062e+01,9.366588,10.842544,6.672720e-09,8.112524e-02,NaN,[2],[4],29.566139,10.842544,NaN,10.842544,9.366588,NaN,8.045374,6.901182,NaN,8.045374,6.901182,NaN,10.842544,9.366588,NaN,1.0,1.0,NaN,1.066646e-03,2.011513e+04,NaN,713.797243,0.239686,NaN,True,True,True,0.000071,0.000071,NaN,1.100606,1.051405,NaN,0.0,0.0,NaN,1.0,1.0,NaN,5.363841,0.020569,NaN,8.0,8.0,NaN,6.409770,8.940780,NaN,NaN,0.0,NaN,NaN,10.842544,NaN,NaN,28.753819,NaN,NaN,1173.557353,NaN,NaN,14.0,NaN,NaN,14.0,NaN,NaN,4.0,NaN,NaN,7.0,NaN,NaN,8.316739,NaN,False,True,False,True,False,False
2,52006,52.736061,44.838073,0.001229,7.545138e-07,1.186103e-10,0.041946,7.980683,False,False,18.514426,15.845164,22.663636,4.194550e-02,14,14,False,5.467227,8.363506e+12,15.845164,18.514426,2.350102e-11,2.315802e-15,NaN,None,None,36.502449,18.514426,NaN,18.514426,15.845164,NaN,13.965002,11.906630,NaN,13.965002,11.906630,NaN,18.514426,15.845164,NaN,0.0,0.0,NaN,7.545138e-07,1.186103e-10,NaN,2622.227264,6723.088132,NaN,False,False,True,0.000071,0.000071,NaN,1.367399,1.271120,NaN,0.0,0.0,NaN,1.0,1.0,NaN,14.154023,22.663636,NaN,8.0,8.0,NaN,4.977686,5.467227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,False
4,52008,37.260339,28.643440,0.000106,1.043206e+02,3.971741e-01,0.136770,0.528105,False,False,12.590454,20.964323,2.219278,2.482534e-16,14,14,False,8.486275,8.805871e+08,12.590454,20.964323,1.021840e-03,1.796025e-05,NaN,[2],None,52.226539,12.590454,NaN,12.590454,20.964323,NaN,9.384955,16.045837,NaN,9.384955,16.045837,NaN,12.590454,20.964323,NaN,1.0,0.0,NaN,1.043206e+02,3.971741e

# Compute redshifts [defunct now]

In [ ]:
# ── Build age(z) ↔ z(age) interpolators using Planck18 ────────────────────
z_grid    = np.linspace(0, 20, 10_000)
t_grid    = Planck18.age(z_grid).to(u.Myr).value       # Myr

age_to_z  = interp1d(t_grid[::-1], z_grid[::-1],       # t decreases with z
                    bounds_error=False, fill_value=(20, 0))

for pop_name in pop_names:
    Z_form    = pop_dfs[pop_name]['Metallicity@ZAMS(1)'].values          # metallicity at formation
    t_delay   = pop_dfs[pop_name]['Time@DCO'].values + pop_dfs[pop_name]['Coalescence_Time'].values                     # delay time in Myr

    # ── Langer & Norman 2006: invert Z_mean(z) = 0.035 * 10^(-0.23 * z) ──────
    # → z_form = log10(Z / 0.035) / -0.23
    mu0, muz  = 0.035, -0.23
    z_form    = np.log10(Z_form / mu0) / muz
    z_form    = np.clip(z_form, 0, 20)

    # ── Merger redshift: t_merge = t(z_form) + t_delay ────────────────────────
    t_form    = Planck18.age(z_form).to(u.Myr).value
    t_merge   = t_form + t_delay

    t_hubble  = Planck18.age(0).to(u.Myr).value
    merges    = t_merge < t_hubble

    z_merger  = age_to_z(t_merge)
    z_merger[~merges] = np.nan

    pop_dfs[pop_name]['z_form']   = z_form
    pop_dfs[pop_name]['z_merger'] = z_merger

In [ ]:
print("Number of BBH mergers in each population:", {pop_name: (pop_df['BBH'] & pop_df['Merges_Hubble_Time']).sum() for pop_name, pop_df in pop_dfs.items()})

Number of BBH mergers in each population: {'z77': 42650}


# Compute Spins


We store the spin of a given star $\omega$ prior to supernova. 

Using this, we can compute the angular momentum stored in the core, 

$$ 
J_{\rm core} = I_{\rm core}  \omega 
$$ 

and the envelope, 

$$ 
J_{\rm shell} = I_{\rm shell}  \omega, 
$$ 

where the moments of inertia are 

$$ 
I_{\rm core} = 0.21 M_{\rm core} R_{\rm core}^2 
$$ 

and 

$$ 
I_{\rm shell} = 0.1 M_{\rm shell} R_{\rm *}^2. 
$$ 


The angular momentum of the final remnant depends on the amount of fallback material. The fraction of material from the shell that falls back onto the compact remnant rather than being ejected is given by:
$$
f_{\rm fallback} = \frac{M_{\rm remnant} - M_{\rm core}}{M_{\rm *} - M_{\rm core}}.
$$

The angular momentum of the remnant is then
is inferred using the star's angular momentum, such that
$$
J_{\rm remnant} = J_{\rm core} + (f_{\rm fallback} * J_{\rm shell}) .
$$

and 

It is assumed that the angular momentum in the core, $ J_{\rm core} = I_{\rm core}  \omega $ remains constant after the supernova, and so the final dimensionless spin can be calculated as

$$
a = \frac{c J_{\rm remnant}}{G M_{\rm remnant}^2}.
$$


If the final spin exceeds 1, we limit it to 1. 

We also impose a limit on the maximum spin from the critical rotation rate (Marchant+ 2023)

$$
a_{\rm max} = 10^{c_0 + c_1 x + c_2 x^2},
$$

where $x = \log_{10}({M_{\rm BH}}/M_{\odot})$, and 
$$
c_0 = 0.13 \sqrt{z} \\
c_1 = 0.74 - 0.19 \sqrt{z} \\
c_2 = -0.58 +0.12 \sqrt{z},
$$
with $z = \log_{10}(0.1 Z_\odot / Z)$.

In [ ]:
ZSOL_ASPLUND = 0.0142

def calculate_dimensionless_spin(m, r, m_remnant, m_core, r_core, omega, metallicity, st):
    m_shell = m - m_core
    j_core = 0.21 * m_core * (r_core**2) * omega
    j_shell = 0.1 * m_shell * (r**2) * omega

    fallback_fraction = np.maximum((m_remnant - m_core)/m_shell, 0)
    fallback_fraction = np.nan_to_num(fallback_fraction, nan=0.0, posinf=0.0, neginf=0.0) # For stars with no shell, fallback fraction becomes NaN or inf. If so, ignore the shell angular momentum.
    fallback_fraction = np.minimum(fallback_fraction, 1.0) # Cap fallback fraction at 1.0

    a = (c.c * (j_core + (fallback_fraction*j_shell))/ (c.G * (m_remnant)**2)).to(u.dimensionless_unscaled)

    zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))
    c_0 = 0.13 * zz
    c_1 = 0.74 - 0.19*zz
    c_2 = -0.58 + 0.12*zz
    xx = np.log10(m_remnant.value)
    a_max = 10**(c_0 + (c_1*xx) + (c_2 * xx**2))

    J_max = a_max * c.G * (m)**2 / c.c
    a_max_bh = (c.c * J_max / (c.G * (m_remnant)**2)).to(u.dimensionless_unscaled)
    a[a>a_max_bh] = a_max_bh[a>a_max_bh]

    a[a>1.0] = 1.0 # set highly rotating cores to have maximal spin
    a[st < 7] = 0.0 # set non He-burning stars to have 0 spin

    return a, fallback_fraction
        
def return_spin_df(pop_df, dco_type=None, mergers=True):
    
    if dco_type is not None:
        pop_df = pop_df[pop_df[dco_type]]
    
    if mergers:
        pop_df = pop_df[pop_df["Merges_Hubble_Time"]]
        
    dco_1_mask = (pop_df['SemiMajorAxis<SN_Primary'] > 0)
    dco_both_1_mask = (pop_df['SemiMajorAxis<SN_Both'] > 0) & ~(pop_df['SemiMajorAxis<SN_Primary'] > 0) # Only include if the primary SN has not been recorded

    dco_2_mask = (pop_df['SemiMajorAxis<SN_Secondary'] > 0) 
    dco_both_2_mask = (pop_df['SemiMajorAxis<SN_Both'] > 0) & ~(pop_df['SemiMajorAxis<SN_Secondary'] > 0)# Only include if the secondary SN has not been recorded
    
    metallicity = pop_df['Metallicity@ZAMS(1)'].values
    
    m1 = pop_df["Mass_Total@CO(SN)_Primary"].values * u.M_sun
    r1 = pop_df["Radius_Total@CO(SN)_Primary"].values * u.R_sun
    m1_remnant = pop_df["Mass@DCO(1)"].values * u.M_sun
    m1_core = pop_df["Mass_CO_Core@CO(SN)_Primary"].values * u.M_sun
    r1_core = pop_df["Radius_Core@CO(SN)_Primary"].values * u.R_sun
    period_orb1 = pop_df["Orbital_Period<SN_Primary"].values * u.d
    st1 = pop_df['Stellar_Type_Prev(SN)_Primary']
    omega_orb_1 = (2 * np.pi / period_orb1).to(1/u.yr)

    j1 = pop_df['Ang_Momentum(SN)_Primary'].values * u.Msun * u.AU**2 / u.yr
    m1_shell = m1 - m1_core
    I1 = (0.21 * m1_core * (r1_core**2) + 0.1 * m1_shell * (r1**2))
    omega1 = (j1 / I1).to(1/u.yr)       
    
    
    
    m2 = pop_df["Mass_Total@CO(SN)_Secondary"].values * u.M_sun
    r2 = pop_df["Radius_Total@CO(SN)_Secondary"].values * u.R_sun
    m2_remnant = pop_df["Mass@DCO(2)"].values * u.M_sun
    m2_core = pop_df["Mass_CO_Core@CO(SN)_Secondary"].values * u.M_sun
    r2_core = pop_df["Radius_Core@CO(SN)_Secondary"].values * u.R_sun
    period_orb2 = pop_df["Orbital_Period<SN_Secondary"].values * u.d
    st2 = pop_df['Stellar_Type_Prev(SN)_Secondary']
    omega_orb_2 = (2 * np.pi / period_orb2).to(1/u.yr)

    j2 = pop_df['Ang_Momentum(SN)_Secondary'].values * u.Msun * u.AU**2 / u.yr
    m2_shell = m2 - m2_core
    I2 = (0.21 * m2_core * (r2_core**2) + 0.1 * m2_shell * (r2**2))
    omega2 = (j2 / I2).to(1/u.yr)      
    
    
    m3 = pop_df["Mass_Total@CO(SN)_Both"].values * u.M_sun
    r3 = pop_df["Radius_Total@CO(SN)_Both"].values * u.R_sun
    m3_remnant = pop_df["Mass@DCO(1)"].values * u.M_sun
    m3_core = pop_df["Mass_CO_Core@CO(SN)_Both"].values * u.M_sun
    r3_core = pop_df["Radius_Core@CO(SN)_Both"].values * u.R_sun
    period_orb3 = pop_df["Orbital_Period<SN_Both"].values * u.d
    st3 = pop_df['Stellar_Type_Prev(SN)_Both']
    omega_orb_3 = (2 * np.pi / period_orb3).to(1/u.yr)

    j3 = pop_df['Ang_Momentum(SN)_Both'].values * u.Msun * u.AU**2 / u.yr
    m3_shell = m3 - m3_core
    I3 = (0.21 * m3_core * (r3_core**2) + 0.1 * m3_shell * (r3**2))
    omega3 = (j3 / I3).to(1/u.yr)      
        
    a1, fb1 = calculate_dimensionless_spin(m1, r1, m1_remnant, m1_core, r1_core, omega1, metallicity, st1)
    a2, fb2 = calculate_dimensionless_spin(m2, r2, m2_remnant, m2_core, r2_core, omega2, metallicity, st2)
    a3, fb3 = calculate_dimensionless_spin(m3, r3, m3_remnant, m3_core, r3_core, omega3, metallicity, st3)
    
    
    # Also compute the spin assuming synchronicity
    a1_orb, fb1 = calculate_dimensionless_spin(m1, r1, m1_remnant, m1_core, r1_core, omega_orb_1, metallicity, st1)
    a2_orb, fb2 = calculate_dimensionless_spin(m2, r2, m2_remnant, m2_core, r2_core, omega_orb_2, metallicity, st2)
    a3_orb, fb3 = calculate_dimensionless_spin(m3, r3, m3_remnant, m3_core, r3_core, omega_orb_3, metallicity, st3)
    
    pop_df['a1'] = np.nan
    pop_df['a2'] = np.nan
    
    pop_df['a1_orb'] = np.nan
    pop_df['a2_orb'] = np.nan

    pop_df.loc[dco_1_mask, "a1"] = a1[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "a1"] = a3[dco_both_1_mask]
    pop_df.loc[dco_2_mask, "a2"] = a2[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "a2"] = a3[dco_both_2_mask]
    
    pop_df.loc[dco_1_mask, "a1_orb"] = a1_orb[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "a1_orb"] = a3_orb[dco_both_1_mask]
    pop_df.loc[dco_2_mask, "a2_orb"] = a2_orb[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "a2_orb"] = a3_orb[dco_both_2_mask]
    
    pop_df['fallback_1'] = np.nan
    pop_df['fallback_2'] = np.nan
    
    pop_df.loc[dco_1_mask, "fallback_1"] = fb1[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "fallback_1"] = fb3[dco_both_1_mask]
    
    pop_df.loc[dco_2_mask, "fallback_2"] = fb2[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "fallback_2"] = fb3[dco_both_2_mask]

    a1, a2, m1, m2 = pop_df["a1"], pop_df["a2"], pop_df["Mass@DCO(1)"], pop_df["Mass@DCO(2)"]
    iota1, iota2, iota2_che = pop_df["SN_Orbit_Inclination_Angle_Primary"], pop_df["SN_Orbit_Inclination_Angle_Secondary"], pop_df["SN_Orbit_Inclination_Angle_Both"]
    iota2 = np.where(np.isnan(iota2), iota2_che, iota2) # If the secondary SN inclination is not recorded, use the "Both" value instead (which should be the same for both SNe)

    pop_df['iota1'] = iota1
    pop_df['iota2'] = iota2

    chi_eff_inclination = ((m1 * a1 * np.cos(iota1)) + (m2 * a2 * np.cos(iota2))) / (m1 + m2)
    pop_df['chi_eff'] = chi_eff_inclination
    
    a1_orb, a2_orb = pop_df["a1_orb"], pop_df["a2_orb"]
    pop_df['chi_eff_orb'] = ((m1 * a1_orb * np.cos(iota1)) + (m2 * a2_orb * np.cos(iota2))) / (m1 + m2)
    
    return pop_df[(~np.isnan(pop_df["a1"]) * ~np.isnan(pop_df["a2"]))]

In [ ]:
pop_spin_dfs = {}
# pop_spin_omega_dfs = {}

for pop_name in pop_names:
    pop_df = pop_dfs[pop_name]
    
    use_omega_orb = False
    if pop_name == 'notides':
        use_omega_orb = True

    pop_spin_dfs[pop_name] = return_spin_df(pop_df, use_omega_orb=use_omega_orb, mergers=False)

    pop_spin_dfs[pop_name].to_csv(data_path + pop_name+'_spin_df.csv', index=False) 
    print("Saved to ", data_path + pop_name +'_spin_df.csv')
    print()


/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/tmp/ipykernel_2382825/2538074320.py:14: RuntimeWarning: invalid value encountered in sqrt
  zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))


Saved to  data_files/z77_spin_df.csv



# Compute Spins as per Bavera+21 Fits

In [ ]:
def function_f_Bavera21(mWR, c1, c2, c3):
    """
    calculates the f() function in Bavera et al (2021) based on given coefficients 
    m_WR with units using astropy
    """

    top = -c1
    bottom = c2 + np.exp(-c3*mWR)

    f = top/bottom
    return f


def calculate_alpha_beta_Bavera21(mWR):
    """ returns alpha and beta for the Bavera et al. (2021) spin calculation """
    # numerical coefficients form text below Eq 2
    # we use the values at helium depletion, since we later on use the C/O core mass. 
    c1_alpha, c2_alpha, c3_alpha =  0.059305, 0.035552, 0.270245
    c1_beta,  c2_beta, c3_beta   =  0.026960, 0.011001, 0.420739

    alpha = function_f_Bavera21(mWR, c1_alpha, c2_alpha, c3_alpha)
    beta  = function_f_Bavera21(mWR, c1_beta,  c2_beta,  c3_beta)
    return alpha, beta

In [ ]:
ZSOL_ASPLUND = 0.0142

def calculate_dimensionless_spin(m, m_core, m_shell, m_remnant, PeriodPreSN, metallicity, st):
    alpha, beta = calculate_alpha_beta_Bavera21(m.value)
    first_term = (alpha* (np.log10(PeriodPreSN.value)**2)) 
    second_term =  (beta * np.log10(PeriodPreSN.value))  
    a = first_term + second_term
    a[a<0.0] = 0.0

    fallback_fraction = np.maximum((m_remnant - m_core)/m_shell, 0)
    fallback_fraction = np.nan_to_num(fallback_fraction, nan=0.0, posinf=0.0, neginf=0.0) # For stars with no shell, fallback fraction becomes NaN or inf. If so, ignore the shell angular momentum.
    fallback_fraction = np.minimum(fallback_fraction, 1.0) # Cap fallback fraction at 1.0

    zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))
    c_0 = 0.13 * zz
    c_1 = 0.74 - 0.19*zz
    c_2 = -0.58 + 0.12*zz
    xx = np.log10(m_remnant.value)
    a_max = 10**(c_0 + (c_1*xx) + (c_2 * xx**2))

    J_max = a_max * c.G * (m)**2 / c.c
    a_max_bh = (c.c * J_max / (c.G * (m_remnant)**2)).to(u.dimensionless_unscaled)
    a[a>a_max_bh] = a_max_bh[a>a_max_bh]

    a[a>1.0] = 1.0 # set highly rotating cores to have maximal spin
    a[st < 7] = 0.0 # set non He-burning stars to have 0 spin

    return a, fallback_fraction
        
def return_spin_df(pop_df, dco_type=None, use_omega_orb=False, mergers=True):
    
    if dco_type is not None:
        pop_df = pop_df[pop_df[dco_type]]
    
    if mergers:
        pop_df = pop_df[pop_df["Merges_Hubble_Time"]]
        
    dco_1_mask = (pop_df['SemiMajorAxis<SN_Primary'] > 0)
    dco_both_1_mask = (pop_df['SemiMajorAxis<SN_Both'] > 0) & ~(pop_df['SemiMajorAxis<SN_Primary'] > 0) # Only include if the primary SN has not been recorded

    dco_2_mask = (pop_df['SemiMajorAxis<SN_Secondary'] > 0) 
    dco_both_2_mask = (pop_df['SemiMajorAxis<SN_Both'] > 0) & ~(pop_df['SemiMajorAxis<SN_Secondary'] > 0)# Only include if the secondary SN has not been recorded
    
    metallicity = pop_df['Metallicity@ZAMS(1)'].values
    
    m1 = pop_df["Mass_Total@CO(SN)_Primary"].values * u.M_sun
    r1 = pop_df["Radius_Total@CO(SN)_Primary"].values * u.R_sun
    m1_remnant = pop_df["Mass@DCO(1)"].values * u.M_sun
    m1_core = pop_df["Mass_CO_Core@CO(SN)_Primary"].values * u.M_sun
    r1_core = pop_df["Radius_Core@CO(SN)_Primary"].values * u.R_sun
    period_orb1 = pop_df["Orbital_Period<SN_Primary"].values * u.d
    st1 = pop_df['Stellar_Type_Prev(SN)_Primary']
    omega_orb_1 = (2 * np.pi / period_orb1).to(1/u.yr)
    # omega1 = pop_df["Omega(SN)_Primary"].values / u.yr
    j1 = pop_df['Ang_Momentum(SN)_Primary'].values * u.Msun * u.AU**2 / u.yr
    m1_shell = m1 - m1_core
    I1 = (0.21 * m1_core * (r1_core**2) + 0.1 * m1_shell * (r1**2))
    omega1 = (j1 / I1).to(1/u.yr)
    period1 = (2 * np.pi / omega1).to(u.d)       
    
    
    
    m2 = pop_df["Mass_Total@CO(SN)_Secondary"].values * u.M_sun
    r2 = pop_df["Radius_Total@CO(SN)_Secondary"].values * u.R_sun
    m2_remnant = pop_df["Mass@DCO(2)"].values * u.M_sun
    m2_core = pop_df["Mass_CO_Core@CO(SN)_Secondary"].values * u.M_sun
    r2_core = pop_df["Radius_Core@CO(SN)_Secondary"].values * u.R_sun
    period_orb2 = pop_df["Orbital_Period<SN_Secondary"].values * u.d
    st2 = pop_df['Stellar_Type_Prev(SN)_Secondary']
    omega_orb_2 = (2 * np.pi / period_orb2).to(1/u.yr)
    # omega2 = pop_df["Omega(SN)_Secondary"].values / u.yr
    j2 = pop_df['Ang_Momentum(SN)_Secondary'].values * u.Msun * u.AU**2 / u.yr
    m2_shell = m2 - m2_core
    I2 = (0.21 * m2_core * (r2_core**2) + 0.1 * m2_shell * (r2**2))
    omega2 = (j2 / I2).to(1/u.yr)   
    period2 = (2 * np.pi / omega2).to(u.d)   
    
    
    m3 = pop_df["Mass_Total@CO(SN)_Both"].values * u.M_sun
    r3 = pop_df["Radius_Total@CO(SN)_Both"].values * u.R_sun
    m3_remnant = pop_df["Mass@DCO(1)"].values * u.M_sun
    m3_core = pop_df["Mass_CO_Core@CO(SN)_Both"].values * u.M_sun
    r3_core = pop_df["Radius_Core@CO(SN)_Both"].values * u.R_sun
    period_orb3 = pop_df["Orbital_Period<SN_Both"].values * u.d
    st3 = pop_df['Stellar_Type_Prev(SN)_Both']
    omega_orb_3 = (2 * np.pi / period_orb3).to(1/u.yr)
    # omega3 = pop_df["Omega(SN)_Both"].values / u.yr
    j3 = pop_df['Ang_Momentum(SN)_Both'].values * u.Msun * u.AU**2 / u.yr
    m3_shell = m3 - m3_core
    I3 = (0.21 * m3_core * (r3_core**2) + 0.1 * m3_shell * (r3**2))
    omega3 = (j3 / I3).to(1/u.yr)     
    period3 = (2 * np.pi / omega3).to(u.d)
    

    a1, fb1 = calculate_dimensionless_spin(m1, m1_core, m1_shell, m1_remnant, period1, metallicity, st1)
    a2, fb2 = calculate_dimensionless_spin(m2, m2_core, m2_shell, m2_remnant, period2, metallicity, st2)
    a3, fb3 = calculate_dimensionless_spin(m3, m3_core, m3_shell, m3_remnant, period3, metallicity, st3)
    
    a1_orb, fb1 = calculate_dimensionless_spin(m1, m1_core, m1_shell, m1_remnant, period_orb1, metallicity, st1)
    a2_orb, fb2 = calculate_dimensionless_spin(m2, m2_core, m2_shell, m2_remnant, period_orb2, metallicity, st2)
    a3_orb, fb3 = calculate_dimensionless_spin(m3, m3_core, m3_shell, m3_remnant, period_orb3, metallicity, st3)

    pop_df['a1'] = np.nan
    pop_df['a2'] = np.nan

    pop_df.loc[dco_1_mask, "a1"] = a1[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "a1"] = a3[dco_both_1_mask]
    
    pop_df.loc[dco_2_mask, "a2"] = a2[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "a2"] = a3[dco_both_2_mask]
    
    
    pop_df['a1_orb'] = np.nan
    pop_df['a2_orb'] = np.nan

    pop_df.loc[dco_1_mask, "a1_orb"] = a1_orb[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "a1_orb"] = a3_orb[dco_both_1_mask]
    
    pop_df.loc[dco_2_mask, "a2_orb"] = a2_orb[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "a2_orb"] = a3_orb[dco_both_2_mask]
    
    
    pop_df['fallback_1'] = np.nan
    pop_df['fallback_2'] = np.nan
    
    pop_df.loc[dco_1_mask, "fallback_1"] = fb1[dco_1_mask]
    pop_df.loc[dco_both_1_mask, "fallback_1"] = fb3[dco_both_1_mask]
    
    pop_df.loc[dco_2_mask, "fallback_2"] = fb2[dco_2_mask]
    pop_df.loc[dco_both_2_mask, "fallback_2"] = fb3[dco_both_2_mask]

    a1, a2, m1, m2 = pop_df["a1"], pop_df["a2"], pop_df["Mass@DCO(1)"], pop_df["Mass@DCO(2)"]
    iota1, iota2, iota2_che = pop_df["SN_Orbit_Inclination_Angle_Primary"], pop_df["SN_Orbit_Inclination_Angle_Secondary"], pop_df["SN_Orbit_Inclination_Angle_Both"]
    iota2 = np.where(np.isnan(iota2), iota2_che, iota2) # If the secondary SN inclination is not recorded, use the "Both" value instead (which should be the same for both SNe)

    pop_df['iota1'] = iota1
    pop_df['iota2'] = iota2

    chi_eff_inclination = ((m1 * a1 * np.cos(iota1)) + (m2 * a2 * np.cos(iota2))) / (m1 + m2)
    pop_df['chi_eff'] = chi_eff_inclination
    
    a1_orb, a2_orb = pop_df["a1_orb"], pop_df["a2_orb"]
    pop_df['chi_eff_orb'] = ((m1 * a1_orb * np.cos(iota1)) + (m2 * a2_orb * np.cos(iota2))) / (m1 + m2)
    
    return pop_df[(~np.isnan(pop_df["a1"]) * ~np.isnan(pop_df["a2"]))]

In [ ]:
pop_spin_dfs = {}

for pop_name in pop_names:
    pop_df = pop_dfs[pop_name]
    
    use_omega_orb = False
    if pop_name == 'notides':
        use_omega_orb = True

    pop_spin_dfs[pop_name] = return_spin_df(pop_df, use_omega_orb=use_omega_orb, mergers=False)

    pop_spin_dfs[pop_name].to_csv(data_path + pop_name+'_bavera_spin_df.csv', index=False) 
    print("Saved to ", data_path + pop_name +'_bavera_spin_df.csv')
    print()


/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/tmp/ipykernel_3006257/1310836178.py:14: RuntimeWarning: invalid value encountered in sqrt
  zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))
/tmp/ipykernel_3006257/1310836178.py:5: RuntimeWarning: divide by zero encountered in log10
  first_term = (alpha* (np.log10(PeriodPreSN.value)**2))
/tmp/ipykernel_3006257/1310836178.py:6: RuntimeWarning: divide by zero encountered in log10
  second_term =  (beta * np.log10(PeriodPreSN.value))
/tmp/ipykernel_3006257/1310836178.py:7: RuntimeWarning: invalid value encountered in add
  a = first_term + second_term


Saved to  data_files/notides_bavera_spin_df.csv



/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/tmp/ipykernel_3006257/1310836178.py:14: RuntimeWarning: invalid value encountered in sqrt
  zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))
/tmp/ipykernel_3006257/1310836178.py:5: RuntimeWarning: divide by zero encountered in log10
  first_term = (alpha* (np.log10(PeriodPreSN.value)**2))
/tmp/ipykernel_3006257/1310836178.py:6: RuntimeWarning: divide by zero encountered in log10
  second_term =  (beta * np.log10(PeriodPreSN.value))
/tmp/ipykernel_3006257/1310836178.py:7: RuntimeWarning: invalid value encountered in add
  a = first_term + second_term


Saved to  data_files/realistic_bavera_spin_df.csv



/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/vkapil1/.conda/envs/tides/lib/python3.9/site-packages/astropy/units/quantity.py:671: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/tmp/ipykernel_3006257/1310836178.py:14: RuntimeWarning: invalid value encountered in sqrt
  zz = np.sqrt(np.log10(0.1 * ZSOL_ASPLUND/ metallicity))


Saved to  data_files/perfect_bavera_spin_df.csv

